# Differential expression analysis of transcriptomic data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garridolab/de-transcriptomics-practical/blob/main/DE_analysis_PBMC_IFNb.ipynb)

### Practical: PBMCs stimulated with interferon-β (IFN-β)

In this practical you will run a full **differential expression (DE) analysis** of RNA-seq count data, from the raw count table through to a list of differentially expressed genes, using **[PyDESeq2](https://github.com/owkin/PyDESeq2)**, a Python implementation of the widely used R/Bioconductor package DESeq2.

**The dataset.** Peripheral blood mononuclear cells (PBMCs) from **8 lupus patients** were split in two. One half was left untreated (`ctrl`) and the other was stimulated with **interferon-β** for 6 hours (`stim`) *(Kang et al., Nature Biotechnology 2018)*. The original experiment is single-cell RNA-seq. We summed the counts of all cells from each donor × condition into **pseudobulk** profiles, so each column of the input file behaves like a bulk RNA-seq library of PBMCs.

| | |
|---|---|
| Input file | `pbmc_ifnb_pseudobulk_counts.csv` (genes × samples, raw integer counts) |
| Samples | 16 = 8 donors × 2 conditions (`ctrl`, `stim`) |
| Design | **paired**: each donor contributes one control and one stimulated sample |
| Question | Which genes change expression in PBMCs after IFN-β stimulation? |

#### Where does the data come from?

* **Original study:** Kang H.M. *et al.* (2018) "Multiplexed droplet single-cell RNA-sequencing using natural genetic variation". *Nature Biotechnology* 36: 89–94. [doi:10.1038/nbt.4042](https://doi.org/10.1038/nbt.4042). Raw data: GEO accession [GSE96583](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE96583).
* **Experiment:** PBMCs from 8 lupus patients were cultured for 6 h with or without recombinant IFN-β. Cells from all donors were pooled and sequenced together on the 10x Genomics Chromium platform (3' UMI counts). Afterwards, each cell was assigned back to its donor from natural genetic variants (SNPs) in the reads, using the *demuxlet* tool. Pooling all donors in the same run means donor and batch are not confounded.
* **Processed version used here:** the curated, cell-type-annotated object `kang_2018.h5ad` (24,673 cells × 15,706 genes) distributed as example data by the scverse/[pertpy](https://pertpy.readthedocs.io) project. It contains 8 annotated cell types: CD4 T, CD8 T, B and NK cells, CD14+ and FCGR3A+ monocytes, dendritic cells and megakaryocytes.
* **What we did to it:** we summed the raw UMI counts of *all* cells from each donor × condition into one profile ("pseudobulk"), giving 16 samples. Genes with zero counts in every sample were dropped. The table was saved as `pbmc_ifnb_pseudobulk_counts.csv`, which the notebook downloads from Google Drive.

**Learning objectives**
1. Understand the structure of a count matrix and do basic quality control.
2. Filter lowly expressed genes and understand why normalisation is needed.
3. Explore sample-to-sample relationships (PCA, distance heatmap) before testing.
4. Fit a negative-binomial GLM with DESeq2 and interpret dispersion estimates.
5. Read and build the standard DE plots: **p-value histogram, MA plot, volcano plot, heatmap and per-gene plots**.
6. Understand why the experimental design (paired vs unpaired) matters.

**Outline:** 0 Setup · 1 Load data · 2 Quality control · 3 Filtering · 4 Normalisation · 5 Exploratory analysis · 6 DE model and dispersion · 7 Results and DE plots · 8 Paired vs unpaired design · 9 Functional enrichment · 10 Export · 11 Exercises

## 0. Setup

The next cell installs every package the notebook needs. Run it once, then **restart the kernel** if any package was newly installed or upgraded. The input counts file is downloaded from Google Drive in section 1.

In [ ]:
%pip install -q "pydeseq2==0.5.4" "pandas>=2.0" "numpy>=1.24" "scipy>=1.10" "matplotlib>=3.7" "seaborn>=0.13" "scikit-learn>=1.3" "adjustText>=1.0" "gseapy>=1.1" "gdown>=5.0"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from adjustText import adjust_text

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

# ---- analysis parameters (change these in the exercises) ----
COUNTS_FILE = "pbmc_ifnb_pseudobulk_counts.csv"
ALPHA = 0.05          # FDR threshold on adjusted p-values
LFC_THRESHOLD = 1.0   # |log2 fold change| threshold (1 = two-fold change)
N_CPUS = 4

# ---- plotting style: one colour per condition, used consistently in every figure ----
COND_COLORS = {"ctrl": "#2a78d6", "stim": "#eb6834"}   # blue = control, orange = IFN-β
COLOR_UP, COLOR_DOWN, COLOR_NS = COND_COLORS["stim"], COND_COLORS["ctrl"], "#c3c2b7"
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#104281", "#3987e5", "#f0efec", "#e66767", "#9b1c1c"])
SEQUENTIAL = LinearSegmentedColormap.from_list("blues", ["#0d366b", "#2a78d6", "#86b6ef", "#e7f0fc"])  # dark = small distance

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK_MUTED, "axes.labelcolor": INK, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.labelsize": 10, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "font.size": 10, "lines.linewidth": 2,
})

def condition_legend(ax, **kw):
    handles = [Line2D([], [], marker="o", ls="", color=c, markersize=8, label=k) for k, c in COND_COLORS.items()]
    ax.legend(handles=handles, title="condition", **kw)

## 1. Load the count matrix and build the sample table

An RNA-seq count matrix has **one row per gene** and **one column per sample**. Each entry is the number of sequencing reads (here, UMIs) assigned to that gene in that sample.

> ⚠️ DESeq2 needs **raw integer counts**. Never give it TPM, FPKM, CPM or log-transformed values: its statistical model relies on the count nature of the data.

In [ ]:
import os
import gdown

GDRIVE_FILE_ID = "1t9QiJaHqE78iUpa1EmMQfaaZiwA3aoe5"
if not os.path.exists(COUNTS_FILE):
    gdown.download(id=GDRIVE_FILE_ID, output=COUNTS_FILE, quiet=False)
else:
    print(f"{COUNTS_FILE} already present, skipping download")

In [ ]:
counts = pd.read_csv(COUNTS_FILE, index_col=0)
print(f"{counts.shape[0]:,} genes x {counts.shape[1]} samples")
counts.iloc[:8, :8]

Sample names follow the pattern `<donor>_<condition>`, so we can build the **sample metadata** (the "colData" in DESeq2 terms) directly from them. In a real project this table usually comes from a separate sample sheet.

In [ ]:
meta = pd.DataFrame(index=counts.columns)
meta.index.name = "sample"
meta["donor"] = meta.index.str.split("_").str[0]
meta["condition"] = pd.Categorical(meta.index.str.split("_").str[1], categories=["ctrl", "stim"])  # first level = reference
meta

In [ ]:
pd.crosstab(meta["donor"], meta["condition"])  # every donor appears once per condition, so the design is paired

## 2. Quality control of the raw counts

Before any modelling, check:
* **Library size** (total counts per sample): very different depths are expected and are corrected by normalisation, but extreme outliers can point to failed libraries.
* **Number of detected genes** per sample.
* The **distribution of counts**: most genes have low counts and a few have very high counts. That skew is why we look at counts on a log scale.

In [ ]:
qc = meta.copy()
qc["library_size_M"] = counts.sum(axis=0) / 1e6
qc["genes_detected"] = (counts > 0).sum(axis=0)
order = qc.sort_values(["condition", "donor"]).index
bar_colors = [COND_COLORS[c] for c in qc.loc[order, "condition"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(order, qc.loc[order, "library_size_M"], color=bar_colors, width=0.75)
axes[0].set(title="Library size", ylabel="total counts (millions)")
axes[1].bar(order, qc.loc[order, "genes_detected"], color=bar_colors, width=0.75)
axes[1].set(title="Genes detected (count > 0)", ylabel="number of genes")
for ax in axes:
    ax.tick_params(axis="x", rotation=90)
    ax.grid(axis="x", visible=False)
condition_legend(axes[0], loc="upper right")
plt.tight_layout(); plt.show()
qc[["library_size_M", "genes_detected"]].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

# (a) one gene, many samples: most genes are lowly expressed
mean_counts = counts.mean(axis=1)
axes[0].hist(np.log10(mean_counts + 1), bins=60, color=INK_MUTED)
axes[0].set(title="Mean count per gene (all genes)", xlabel="log10(mean count + 1)", ylabel="number of genes")

# (b) per-sample distributions of log counts
log_raw = np.log2(counts[order] + 1)
bp = axes[1].boxplot(log_raw.values, patch_artist=True, showfliers=False, widths=0.6, medianprops=dict(color=INK))
for patch, c in zip(bp["boxes"], bar_colors):
    patch.set_facecolor(c); patch.set_alpha(0.8)
axes[1].set_xticks(range(1, len(order) + 1), order, rotation=90)
axes[1].set(title="Raw counts per sample", ylabel="log2(count + 1)")
axes[1].grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

print(f"Genes with zero counts in every sample: {(counts.sum(axis=1) == 0).sum()}")
print(f"Genes with mean count < 1: {(mean_counts < 1).sum():,} ({(mean_counts < 1).mean():.0%})")

**Questions**
* Library sizes differ about five-fold between samples. Could we compare raw counts of a gene directly between `P1015_ctrl` and `P107_ctrl`? Why not?
* Most of the right-hand boxes sit near zero. What does that tell you about the typical gene in this dataset?

## 3. Filtering lowly expressed genes

Genes with very few counts carry almost no statistical information. They cannot reach significance, yet each one still counts as a test in the multiple-testing correction. Removing them up front:
* reduces the multiple-testing burden,
* speeds up the model fit,
* makes the diagnostic plots easier to read.

A common rule (similar to edgeR's `filterByExpr`) is to **keep genes with at least 10 counts in at least *n* samples**, where *n* is the size of the smallest group (8 here). DESeq2 also runs its own *independent filtering* later, so this step is about removing obvious noise, not about being clever.

In [ ]:
MIN_COUNT = 10
MIN_SAMPLES = meta["condition"].value_counts().min()

keep = (counts >= MIN_COUNT).sum(axis=1) >= MIN_SAMPLES
counts_f = counts.loc[keep]
print(f"Kept {keep.sum():,} of {len(keep):,} genes ({keep.mean():.0%}) with >= {MIN_COUNT} counts in >= {MIN_SAMPLES} samples")

fig, ax = plt.subplots(figsize=(7, 3.6))
bins = np.linspace(0, np.log10(mean_counts.max() + 1), 70)
ax.hist(np.log10(mean_counts[~keep] + 1), bins=bins, color=COLOR_NS, label=f"removed (n={(~keep).sum():,})")
ax.hist(np.log10(mean_counts[keep] + 1), bins=bins, color=INK_MUTED, label=f"kept (n={keep.sum():,})")
ax.set(title="Effect of the expression filter", xlabel="log10(mean count + 1)", ylabel="number of genes")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Normalisation: size factors

Raw counts depend on **sequencing depth** (and, more subtly, on RNA composition). DESeq2 uses the **median-of-ratios** method:

1. For each gene, compute the geometric mean across all samples (a "pseudo-reference" sample).
2. For each sample, compute the ratio of every gene's count to that reference.
3. The **size factor** of the sample is the **median** of those ratios.

Normalised counts = raw counts / size factor. Because a median is used, a handful of strongly changing genes (like the interferon response here) do not distort the normalisation, which can happen with simple total-count (CPM) scaling.

> PyDESeq2, like DESeq2, expects a **samples × genes** table, so we transpose the matrix.

In [ ]:
inference = DefaultInference(n_cpus=N_CPUS)
dds = DeseqDataSet(
    counts=counts_f.T,              # samples x genes
    metadata=meta,
    design="~ donor + condition",   # paired design: compare conditions *within* each donor
    refit_cooks=True,
    inference=inference,
    quiet=True,
)
dds.fit_size_factors()
size_factors = dds.obs["size_factors"]

fig, ax = plt.subplots(figsize=(5.5, 4.2))
lib = counts_f.sum(axis=0) / 1e6
ax.scatter(lib, size_factors, c=[COND_COLORS[c] for c in meta["condition"]], s=60, edgecolor="white", linewidth=1.5)
for s in meta.index:
    ax.annotate(meta.loc[s, "donor"], (lib[s], size_factors[s]), xytext=(5, 0), textcoords="offset points", fontsize=8, color=INK_MUTED, va="center")
ax.set(title="Size factors track sequencing depth", xlabel="library size (millions, filtered genes)", ylabel="DESeq2 size factor")
condition_legend(ax, loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
normed = pd.DataFrame(dds.layers["normed_counts"], index=dds.obs_names, columns=dds.var_names).T  # genes x samples

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, mat, title in [(axes[0], counts_f, "Before normalisation (raw)"), (axes[1], normed, "After normalisation (median of ratios)")]:
    bp = ax.boxplot(np.log2(mat[order] + 1).values, patch_artist=True, showfliers=False, widths=0.6, medianprops=dict(color=INK))
    for patch, c in zip(bp["boxes"], bar_colors):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    ax.set_xticks(range(1, len(order) + 1), order, rotation=90)
    ax.set(title=title); ax.grid(axis="x", visible=False)
axes[0].set_ylabel("log2(count + 1)")
plt.tight_layout(); plt.show()

After normalisation the sample medians line up, so counts are now comparable **between samples**. They are still *not* comparable between genes, because longer or more abundant transcripts naturally collect more reads.

## 5. Exploratory analysis: how do samples relate to each other?

Before testing individual genes, look at the samples as a whole. Do replicates cluster together? Is the condition effect the main source of variation? Are there outliers or batch effects (here, donors)?

### 5.1 Variance-stabilising transformation (VST)

For counts, the **variance grows with the mean**. Even after taking `log2(x + 1)`, low-count genes are very noisy, so they would dominate distance-based methods such as PCA. The **VST** returns log2-like values whose variance is roughly independent of the mean.

> VST values are used **only for visualisation and clustering**. The statistical test itself runs on the raw counts (step 6).

In the plot below, each dot is a gene and the orange line is the running median of its standard deviation. A flat line means variance no longer depends on expression level. With these deep pseudobulk libraries, and after filtering, the gain over `log2(x + 1)` is modest and visible mainly for the lowest-expressed genes. In typical bulk RNA-seq with lower depth the difference is much larger.

In [ ]:
dds.vst(use_design=False)   # blind to the design, as recommended for QC
vst = pd.DataFrame(dds.layers["vst_counts"], index=dds.obs_names, columns=dds.var_names).T  # genes x samples
log_norm = np.log2(normed + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, mat, title in [(axes[0], log_norm, "log2(normalised count + 1)"), (axes[1], vst, "VST")]:
    rank = mat.mean(axis=1).rank()
    ax.scatter(rank, mat.std(axis=1), s=3, color=INK_MUTED, alpha=0.25, rasterized=True)
    roll = mat.std(axis=1).groupby(pd.cut(rank, 40)).median()
    ax.plot([iv.mid for iv in roll.index], roll.values, color=COND_COLORS["stim"], label="running median")
    ax.set(title=title, xlabel="rank of mean expression (low → high)", ylim=(0, 3))
    ax.legend(loc="upper right")
axes[0].set_ylabel("standard deviation across samples")
fig.suptitle("Mean–variance relationship", fontweight="bold")
plt.tight_layout(); plt.show()

### 5.2 Principal component analysis (PCA)

PCA summarises the thousands of genes into a few axes that capture the most variance. Following DESeq2's `plotPCA`, we use the **500 most variable genes** after VST. Grey lines join the two samples from the same donor.

In [ ]:
N_TOP = 500
top_var = vst.var(axis=1).sort_values(ascending=False).index[:N_TOP]
X = vst.loc[top_var].T                                  # samples x genes
pca = PCA(n_components=6).fit(X - X.mean())
pcs = pd.DataFrame(pca.transform(X - X.mean())[:, :3], index=X.index, columns=["PC1", "PC2", "PC3"]).join(meta)
ve = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.2, 1.2, 0.8]})
for ax, (a, b) in zip(axes[:2], [("PC1", "PC2"), ("PC2", "PC3")]):
    for d, sub in pcs.groupby("donor"):
        ax.plot(sub[a], sub[b], color=COLOR_NS, lw=1, zorder=1)
    ax.scatter(pcs[a], pcs[b], c=[COND_COLORS[c] for c in pcs["condition"]], s=70, edgecolor="white", linewidth=1.5, zorder=2)
    texts = [ax.text(r[a], r[b], r["donor"], fontsize=8, color=INK_MUTED) for _, r in pcs.iterrows()]
    adjust_text(texts, ax=ax)
    ax.set(xlabel=f"{a} ({ve[int(a[-1]) - 1]:.1f}% variance)", ylabel=f"{b} ({ve[int(b[-1]) - 1]:.1f}% variance)", title=f"PCA: {a} vs {b}")
condition_legend(axes[0], loc="best")
axes[2].bar([f"PC{i}" for i in range(1, 7)], ve, color=INK_MUTED, width=0.7)
axes[2].set(title="Scree plot", ylabel="% variance explained"); axes[2].grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

**Questions**
* Which factor drives PC1? How much variance does it explain?
* Look at PC2/PC3. Do samples from the same donor sit close to each other? What does that suggest about including `donor` in the model?
* In the distance heatmap below, samples first split by condition. Within each condition, do donors P107 and P1039 (the smallest libraries) behave differently?

### 5.3 Sample-to-sample distance heatmap

Euclidean distances between samples (on VST values), clustered hierarchically. Dark cells mean similar samples. The colour bar above the columns shows the condition.

In [ ]:
dist = pd.DataFrame(squareform(pdist(vst.T, metric="euclidean")), index=vst.columns, columns=vst.columns)
g = sns.clustermap(
    dist, cmap=SEQUENTIAL, figsize=(8, 7.5), method="average",
    col_colors=meta["condition"].map(COND_COLORS).rename("condition"),
    row_colors=meta["condition"].map(COND_COLORS).rename("condition"),
    cbar_kws={"label": "Euclidean distance (VST)"}, linewidths=0.5, linecolor="white", dendrogram_ratio=0.12,
    cbar_pos=(1.02, 0.35, 0.025, 0.3),
)
g.ax_heatmap.grid(False)
g.fig.suptitle("Sample-to-sample distances", fontweight="bold", y=1.01)
plt.show()

## 6. Fitting the DESeq2 model

DESeq2 models the count $K_{ij}$ of gene $i$ in sample $j$ with a **negative binomial (NB)** distribution:

$$K_{ij} \sim \mathrm{NB}(\mu_{ij}, \alpha_i), \qquad \mu_{ij} = s_j \, q_{ij}, \qquad \log_2 q_{ij} = \sum_r x_{jr}\,\beta_{ir}$$

* $s_j$: size factor of sample *j* (section 4)
* $\alpha_i$: **dispersion** of gene *i*, i.e. the biological variability beyond Poisson noise ($\mathrm{Var} = \mu + \alpha\mu^2$)
* $\beta_{ir}$: coefficients of the design, here `~ donor + condition`. The `condition` coefficient is the **log2 fold change stim vs ctrl**, estimated *within donors*.

With only a few replicates, per-gene dispersion estimates are noisy. DESeq2 **shares information across genes**: it fits a trend of dispersion against mean expression and *shrinks* each gene's estimate towards that trend. Then, for each gene, it runs a **Wald test** of the null hypothesis that the condition coefficient is zero.

`dds.deseq2()` runs every step: size factors → gene-wise dispersions → trend → shrunken (MAP) dispersions → GLM fit → Cook's-distance outlier handling.

In [ ]:
dds.deseq2()
print("Design matrix columns:", list(dds.obsm["design_matrix"].columns))

### 6.1 Dispersion plot

This plot is the main diagnostic of the DESeq2 fit. Expect:
* gene-wise estimates (grey) scattered around a **decreasing trend** (red): low-count genes are relatively more variable;
* final estimates (blue) pulled towards the trend;
* genes far *above* the trend are kept at their gene-wise value (shown as blue points with an outline). They are likely truly highly variable.

In [ ]:
v = dds.var
fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.scatter(v["_normed_means"], v["genewise_dispersions"], s=3, color=COLOR_NS, alpha=0.7, label="gene-wise estimate", rasterized=True)
ax.scatter(v["_normed_means"], v["dispersions"], s=3, color=COND_COLORS["ctrl"], alpha=0.6, label="final (MAP) estimate", rasterized=True)
out = v["_outlier_genes"]
ax.scatter(v.loc[out, "_normed_means"], v.loc[out, "dispersions"], s=14, facecolor="none", edgecolor=COND_COLORS["ctrl"], linewidth=0.8, label="dispersion outlier (not shrunk)")
srt = v.sort_values("_normed_means")
ax.plot(srt["_normed_means"], srt["fitted_dispersions"], color="#e34948", label="fitted trend")
ax.set(xscale="log", yscale="log", xlabel="mean of normalised counts", ylabel="dispersion", title="Dispersion estimates")
ax.set_ylim(1e-4, v["genewise_dispersions"].max() * 2)   # genes with gene-wise estimate ~0 (no extra-Poisson variation) fall below the axis
ax.legend(markerscale=3, loc="lower left")
plt.tight_layout(); plt.show()

## 7. Results: stimulated vs control

We ask for the contrast **`stim` vs `ctrl`**, so positive log2 fold changes (LFC) mean **higher in IFN-β-stimulated cells**.

The results table contains, for each gene:

| column | meaning |
|---|---|
| `baseMean` | mean of normalised counts across all samples |
| `log2FoldChange` | effect size: log2(stim / ctrl) |
| `lfcSE` | standard error of the LFC |
| `stat` | Wald statistic = LFC / SE |
| `pvalue` | raw p-value of the Wald test |
| `padj` | **Benjamini–Hochberg adjusted p-value (FDR)**. Use this one, not `pvalue`. |

`padj` is `NaN` for genes excluded by **independent filtering** (too low counts to reach significance) or flagged as count outliers.

In [ ]:
ds = DeseqStats(dds, contrast=["condition", "stim", "ctrl"], alpha=ALPHA, inference=inference, quiet=True)
ds.summary()
res = ds.results_df.copy()   # keep the unshrunken results
res.sort_values("padj").head(10)

In [ ]:
def classify(df, alpha=ALPHA, lfc=LFC_THRESHOLD, col="log2FoldChange"):
    sig = (df["padj"] < alpha) & (df[col].abs() > lfc)
    return np.select([sig & (df[col] > 0), sig & (df[col] < 0)], ["up", "down"], default="ns")

res["class"] = classify(res)
n_sig_any = (res["padj"] < ALPHA).sum()
print(f"Genes tested: {res['padj'].notna().sum():,} (padj = NaN for {res['padj'].isna().sum():,})")
print(f"padj < {ALPHA}: {n_sig_any:,} genes")
print(f"padj < {ALPHA} and |LFC| > {LFC_THRESHOLD}: {(res['class'] != 'ns').sum():,} genes "
      f"({(res['class'] == 'up').sum():,} up, {(res['class'] == 'down').sum():,} down in stim)")

### 7.1 P-value histogram

Always look at the distribution of **raw** p-values before trusting the results:
* **Flat** background plus a **peak near 0**: well-behaved test, with true positives in the peak.
* Peak near 1 or a hump in the middle: model misspecification, missing covariates or bad filtering.

Under the null hypothesis p-values are uniform, so the height of the flat part tells you roughly how many genes are truly null.

In [ ]:
pv = res["pvalue"].dropna()
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(pv, bins=40, range=(0, 1), color=INK_MUTED, edgecolor="white", linewidth=0.5)
null_level = pv[pv > 0.5].size / 20   # average bin height in the right half, where nearly all genes are null
ax.axhline(null_level, color=COND_COLORS["stim"], ls="--", lw=1.5, label="approx. null level")
ax.set(title="Distribution of raw p-values", xlabel="raw p-value", ylabel="number of genes")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()
print(f"Rough estimate of the fraction of truly null genes (pi0): {min(1, null_level * 40 / pv.size):.0%}")

### 7.2 Log-fold-change shrinkage

Genes with **low counts** or **high dispersion** can show huge but unreliable LFCs. **Shrinkage** (here the *apeGLM* method) pulls noisy LFCs towards zero while leaving well-measured LFCs almost untouched. Shrunken LFCs are better for **ranking genes and visualisation**. P-values do not change.

In [ ]:
coeff = [c for c in dds.varm["LFC"].columns if c.startswith("condition")][0]
print("Shrinking coefficient:", coeff)
ds.lfc_shrink(coeff=coeff)
res["log2FoldChange_shrunk"] = ds.results_df["log2FoldChange"]
res["lfcSE_shrunk"] = ds.results_df["lfcSE"]
res["class_shrunk"] = classify(res, col="log2FoldChange_shrunk")
res[["baseMean", "log2FoldChange", "log2FoldChange_shrunk", "padj"]].sort_values("padj").head()

### 7.3 MA plot

The **MA plot** shows the log2 fold change (*M*) against mean expression (*A*). Every dot is a gene, and significant genes are coloured.

Compare the two panels. Without shrinkage, low-count genes (left) have the noisiest LFCs, and shrinkage pulls those towards 0 while the well-measured genes keep their effects. Pseudobulk libraries are deep, so the effect is subtle here. Look at the left end of the cloud and at how the number of genes passing the |LFC| threshold changes. With low-depth data or few replicates the difference is dramatic.

In [ ]:
def ma_plot(ax, df, lfc_col, cls_col, title):
    order_ = ["ns", "down", "up"]
    palette = {"ns": COLOR_NS, "up": COLOR_UP, "down": COLOR_DOWN}
    labels = {"ns": "not significant", "up": f"up in stim (n={(df[cls_col] == 'up').sum()})", "down": f"down in stim (n={(df[cls_col] == 'down').sum()})"}
    for k in order_:
        sub = df[df[cls_col] == k]
        ax.scatter(sub["baseMean"], sub[lfc_col], s=4 if k == "ns" else 7, color=palette[k], alpha=0.6 if k == "ns" else 0.85,
                   label=labels[k], rasterized=True)
    ax.axhline(0, color=INK, lw=1)
    for y in (-LFC_THRESHOLD, LFC_THRESHOLD):
        ax.axhline(y, color=INK_MUTED, lw=1, ls="--")
    ax.set(xscale="log", xlabel="mean of normalised counts (A)", ylabel="log2 fold change stim / ctrl (M)", title=title)
    ax.legend(markerscale=2.5, loc="lower right")

ylim = np.nanmax(res["log2FoldChange"].abs()) * 1.05
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
ma_plot(axes[0], res, "log2FoldChange", "class", "MA plot: MLE (unshrunken) LFC")
ma_plot(axes[1], res, "log2FoldChange_shrunk", "class_shrunk", "MA plot: apeGLM-shrunken LFC")
axes[0].set_ylim(-ylim, ylim)
plt.tight_layout(); plt.show()

### 7.4 Volcano plot

The **volcano plot** combines effect size (x: log2 fold change) with statistical evidence (y: −log10 adjusted p-value). Genes in the upper corners have both a large and a significant change. Dashed lines mark the thresholds (`ALPHA`, `LFC_THRESHOLD`).

In [ ]:
N_LABELS = 20
vol = res.dropna(subset=["padj"]).copy()
min_p = vol.loc[vol["padj"] > 0, "padj"].min()
vol["neglog10_padj"] = -np.log10(vol["padj"].clip(lower=min_p))   # avoid log10(0) for extremely significant genes
lfc_col, cls_col = "log2FoldChange_shrunk", "class_shrunk"

fig, ax = plt.subplots(figsize=(9, 6.5))
palette = {"ns": COLOR_NS, "up": COLOR_UP, "down": COLOR_DOWN}
labels = {"ns": "not significant", "up": f"up in stim (n={(vol[cls_col] == 'up').sum()})", "down": f"down in stim (n={(vol[cls_col] == 'down').sum()})"}
for k in ["ns", "down", "up"]:
    sub = vol[vol[cls_col] == k]
    ax.scatter(sub[lfc_col], sub["neglog10_padj"], s=5 if k == "ns" else 10, color=palette[k], alpha=0.6 if k == "ns" else 0.85, label=labels[k], rasterized=True)
ax.axhline(-np.log10(ALPHA), color=INK_MUTED, ls="--", lw=1)
for x in (-LFC_THRESHOLD, LFC_THRESHOLD):
    ax.axvline(x, color=INK_MUTED, ls="--", lw=1)

to_label = pd.concat([
    vol[vol[cls_col] == "up"].sort_values("padj").head(N_LABELS // 2),
    vol[vol[cls_col] == "down"].sort_values("padj").head(N_LABELS // 2),
])
texts = [ax.text(r[lfc_col], r["neglog10_padj"], g, fontsize=8, color=INK) for g, r in to_label.iterrows()]
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color=INK_MUTED, lw=0.5))
ax.set(xlabel="log2 fold change stim / ctrl (shrunken)", ylabel="−log10 adjusted p-value", title="Volcano plot: IFN-β stimulated vs control PBMCs")
ax.legend(markerscale=2, loc="upper left")
plt.tight_layout(); plt.show()

**Questions**
* Look up a few of the labelled up-regulated genes (e.g. *ISG15*, *IFI6*, *IFIT1/3*, *MX1*, *OAS1*). What do they have in common?
* Why is the volcano plot asymmetric?

### 7.5 Heatmap of the top differentially expressed genes

A heatmap of the most significant DE genes across all samples. Values are **VST expression, z-scored per gene** (row): red means above that gene's average and blue means below. Genes and samples are clustered hierarchically.

In [ ]:
N_HEAT = 50
top_genes = res[res["class_shrunk"] != "ns"].sort_values("padj").head(N_HEAT).index
z = vst.loc[top_genes]
z = z.sub(z.mean(axis=1), axis=0).div(z.std(axis=1), axis=0)
lim = np.ceil(np.abs(z.values).max())

g = sns.clustermap(
    z, cmap=DIVERGING, center=0, vmin=-lim, vmax=lim, figsize=(9, 12), method="average", metric="euclidean",
    col_colors=meta["condition"].map(COND_COLORS).rename("condition"),
    cbar_kws={"label": "z-score (VST)"}, yticklabels=True, dendrogram_ratio=(0.12, 0.08),
    cbar_pos=(1.02, 0.4, 0.025, 0.2),
)
g.ax_heatmap.tick_params(axis="y", labelsize=7)
g.ax_heatmap.set_ylabel("")
g.ax_heatmap.grid(False)
g.fig.suptitle(f"Top {N_HEAT} DE genes (by padj)", fontweight="bold", y=1.01)
plt.show()

### 7.6 Per-gene plots

Always look at the raw data behind a result. Below are normalised counts for the top up- and down-regulated genes. Each grey line joins the two samples from **one donor**, so you can see the paired structure the model exploits.

In [ ]:
def gene_panel(genes, ncols=4):
    nrows = int(np.ceil(len(genes) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.1 * nrows), squeeze=False)
    xpos = {"ctrl": 0, "stim": 1}
    for ax, gene in zip(axes.flat, genes):
        y = normed.loc[gene] + 0.5
        for d in meta["donor"].unique():
            s = meta.index[meta["donor"] == d]
            ax.plot([xpos[c] for c in meta.loc[s, "condition"]], y[s], color=COLOR_NS, lw=1, zorder=1)
        ax.scatter([xpos[c] for c in meta["condition"]], y[meta.index], c=[COND_COLORS[c] for c in meta["condition"]],
                   s=45, edgecolor="white", linewidth=1.2, zorder=2)
        ax.set_yscale("log")
        ax.set_xticks([0, 1], ["ctrl", "stim"]); ax.set_xlim(-0.4, 1.4)
        r = res.loc[gene]
        ax.set_title(f"{gene}\nLFC={r['log2FoldChange_shrunk']:.2f}, padj={r['padj']:.1e}", fontsize=9)
        ax.grid(axis="x", visible=False)
    for ax in axes[:, 0]:
        ax.set_ylabel("normalised count + 0.5")
    for ax in list(axes.flat)[len(genes):]:
        ax.set_visible(False)
    plt.tight_layout(); plt.show()

sig = res[res["class_shrunk"] != "ns"].sort_values("padj")
top_up = sig[sig["class_shrunk"] == "up"].index[:4].tolist()
top_down = sig[sig["class_shrunk"] == "down"].index[:4].tolist()
gene_panel(top_up + top_down)

## 8. Why the design matters: paired vs unpaired

Our model was `~ donor + condition`. Each donor acts as its own control, so stable between-donor differences (genetics, age, disease activity…) are **removed** from the error term.

What if we ignored the pairing and fitted `~ condition` instead? Donor variation then ends up in the residual noise, which lowers the power to detect DE genes.

In [ ]:
dds_unpaired = DeseqDataSet(counts=counts_f.T, metadata=meta, design="~ condition", refit_cooks=True, inference=inference, quiet=True)
dds_unpaired.deseq2()
ds_unpaired = DeseqStats(dds_unpaired, contrast=["condition", "stim", "ctrl"], alpha=ALPHA, inference=inference, quiet=True)
ds_unpaired.summary()
res_unpaired = ds_unpaired.results_df

summary = pd.DataFrame({
    "paired (~ donor + condition)": [(res["padj"] < ALPHA).sum(), (res["class"] != "ns").sum()],
    "unpaired (~ condition)": [(res_unpaired["padj"] < ALPHA).sum(), (classify(res_unpaired) != "ns").sum()],
}, index=[f"padj < {ALPHA}", f"padj < {ALPHA} & |LFC| > {LFC_THRESHOLD}"])
summary

In [ ]:
both = pd.DataFrame({"paired": res["padj"], "unpaired": res_unpaired["padj"]}).dropna()
both = -np.log10(both.clip(lower=1e-300))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ax = axes[0]
ax.scatter(both["unpaired"], both["paired"], s=4, color=INK_MUTED, alpha=0.5, rasterized=True)
m = both.max().max()
ax.plot([0, m], [0, m], color=COND_COLORS["stim"], lw=1.5, label="y = x")
ax.set(xscale="symlog", yscale="symlog", xlabel="−log10 padj, unpaired", ylabel="−log10 padj, paired", title="Evidence for DE: paired vs unpaired")
ax.legend(loc="upper left")

ax = axes[1]
x = np.arange(len(summary))
w = 0.38
ax.bar(x - w / 2, summary.iloc[:, 0], width=w, color=INK, label=summary.columns[0])
ax.bar(x + w / 2, summary.iloc[:, 1], width=w, color=COLOR_NS, label=summary.columns[1])
for i in range(len(summary)):
    ax.text(i - w / 2, summary.iloc[i, 0], f"{summary.iloc[i, 0]:,}", ha="center", va="bottom", fontsize=9, color=INK)
    ax.text(i + w / 2, summary.iloc[i, 1], f"{summary.iloc[i, 1]:,}", ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xticks(x, summary.index); ax.set(ylabel="number of DE genes", title="Number of significant genes")
ax.grid(axis="x", visible=False); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()
print(f"{(both['paired'] > both['unpaired']).mean():.0%} of genes have stronger evidence (smaller padj) with the paired design.")

**Interpretation.** Accounting for donors gives most genes stronger evidence, and several hundred extra genes become significant. Most of these have **moderate** fold changes, which is why the count barely moves once we also require |LFC| > 1. Very large effects, like the interferon response itself, are detected either way. The design matters most for subtle effects, and it matters more the larger the between-individual variation is.

## 9. Functional enrichment (optional, needs internet)

A list of hundreds of genes is hard to interpret one by one. **Over-representation analysis** asks whether genes from known pathways or GO terms appear in the DE list more often than expected by chance. Here we send the up-regulated genes to the [Enrichr](https://maayanlab.cloud/Enrichr/) web service through `gseapy`.

> In a formal analysis, use the set of *tested* genes as the background (universe), not the whole genome.

In [ ]:
import gseapy as gp

up_genes = res.index[res["class_shrunk"] == "up"].tolist()
background = res.index[res["padj"].notna()].tolist()
try:
    enr = gp.enrichr(gene_list=up_genes, gene_sets=["GO_Biological_Process_2023"], organism="human",
                     background=background, outdir=None, cutoff=1.0)
    top_terms = enr.results.sort_values("Adjusted P-value").head(15).copy()
    top_terms["term"] = top_terms["Term"].str.replace(r"\s*\(GO:\d+\)", "", regex=True)
    top_terms["neglog10_padj"] = -np.log10(top_terms["Adjusted P-value"])
    top_terms["n_genes"] = top_terms["Genes"].str.split(";").str.len()   # DE genes annotated to the term

    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(top_terms["term"][::-1], top_terms["neglog10_padj"][::-1], color=COLOR_UP, height=0.7)
    for i, (v_, n) in enumerate(zip(top_terms["neglog10_padj"][::-1], top_terms["n_genes"][::-1])):
        ax.text(v_, i, f"  {n} genes", va="center", fontsize=8, color=INK_MUTED)
    ax.set(xlabel="−log10 adjusted p-value", title=f"GO Biological Process enriched among {len(up_genes)} up-regulated genes")
    ax.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()
    display(top_terms[["Term", "n_genes", "Adjusted P-value", "Genes"]].head(10))
except Exception as e:
    plt.close("all")
    print(f"Enrichment skipped (no internet connection or Enrichr unavailable): {e}")

## 10. Export the results

In [ ]:
out = res.drop(columns=["class"]).rename(columns={"class_shrunk": "DE_class"}).sort_values("padj")
out.index.name = "gene"
out.to_csv("de_results_stim_vs_ctrl.csv")
normed.round(2).to_csv("normalized_counts.csv")
print("Wrote de_results_stim_vs_ctrl.csv and normalized_counts.csv")
out.head()

## 11. Summary and exercises

**What we did**
1. QC'd raw counts (library sizes, detected genes, distributions).
2. Filtered lowly expressed genes.
3. Normalised with DESeq2 size factors (median of ratios).
4. Explored samples with VST, PCA and a distance heatmap. Condition dominates, with a strong donor effect on top.
5. Fitted a negative-binomial GLM (`~ donor + condition`) and checked the dispersion plot.
6. Tested stim vs ctrl and inspected the p-value histogram, MA plot (raw vs shrunken LFC), volcano plot, heatmap and per-gene plots.
7. Showed that modelling the paired design increases power.
8. Interpreted the up-regulated genes with GO enrichment. The interferon / antiviral response dominates, as expected.

**A caveat.** These are pseudobulk profiles of *all* PBMCs together. Some apparent "DE" may come from changes in **cell-type composition**, or come from one cell type only (e.g. monocytes). For single-cell data, the usual approach is to build pseudobulks **per cell type** and run this same workflow on each.

**Exercises**
1. Change `LFC_THRESHOLD` to `0` and then to `2`. How do the number of DE genes and the volcano plot change?
2. Change `MIN_COUNT` to `0` (no filtering). Re-run and compare the p-value histogram and the number of tested genes.
3. **Power and replicates:** keep only 3 donors (e.g. `meta = meta[meta.donor.isin(["P1015", "P1016", "P1244"])]` and subset `counts` to the same columns), then re-run. How many DE genes do you find now?
4. Pick a gene with a large unshrunken LFC but small `baseMean` from the MA plot. Plot it with `gene_panel([...])`. Do you trust it? How does shrinkage treat it?
5. Run the enrichment on the **down-regulated** genes. What biology comes out?
6. *(Advanced)* Test the contrast in the other direction (`["condition", "ctrl", "stim"]`). What changes in the results table?